
# USGS Groundwater & Stream Discharge — Cleaning & ArcGIS Export (with Decades)

This notebook helps you:
- Load the two tab-delimited text files you provided (groundwater & stream discharge).
- Clean and standardize field names, dates, coordinates, and data types.
- Validate coordinates and set a proper CRS (NAD83 → EPSG:4269, WGS84 → EPSG:4326, NAD27 → EPSG:4267).
- **Assign a decade** (e.g., `1950s`) to each measurement.
- Export **ArcGIS-ready** outputs as CSV, Shapefile, and GeoPackage, including **per-decade** layers.

> Tip: Run the notebook top-to-bottom. Adjust the file paths in the first code cell if your files are in a different folder.



## 0) Environment (run once)
If you're running this on a fresh environment, install the packages below.
(You can skip if they are already installed.)


In [ ]:

# If needed, uncomment and run:
# %pip install pandas geopandas pyproj shapely



## 1) Configure input/output
Set the file paths for your two input files and choose your outputs.


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# EDIT THESE if your files live elsewhere:
GROUNDWATER_PATH = Path('/mnt/data/usgs_1890_1980.txt')   # groundwater (tab-delimited)
STREAM_PATH      = Path('/mnt/data/usgs_1820_1980_sd.txt') # stream discharge (tab-delimited)
# <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<

# Output directory and names
OUT_DIR = Path('./arcgis_outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Choose export formats
EXPORT_CSV         = True
EXPORT_SHAPEFILE   = True      # field name limit of 10 chars applies
EXPORT_GEOPACKAGE  = True

# For Shapefile and GeoPackage
LAYER_NAME_GW = 'gw_sites'
LAYER_NAME_SD = 'sd_sites'

print('Groundwater file:', GROUNDWATER_PATH.resolve())
print('Stream file     :', STREAM_PATH.resolve())
print('Outputs folder  :', OUT_DIR.resolve())



## 2) Helper functions
Generic utilities to load, clean and export data.


In [ ]:

import re
from datetime import datetime
from typing import Tuple, Optional, Dict

def load_tab_file(path: Path) -> pd.DataFrame:
    """Load a tab-delimited file with robust options."""
    df = pd.read_csv(path, sep='\t', dtype=str, na_values=['', 'NA', 'NaN', 'NULL', '--'])
    # Strip whitespace from column names and values
    df.columns = [c.strip() for c in df.columns]
    for c in df.columns:
        if df[c].dtype == object:
            df[c] = df[c].str.strip()
    return df

def standardize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """Lowercase, snake_case, and shorten overly long names for shapefile safety."""
    rename_map = {}
    for c in df.columns:
        cc = c.strip()
        cc = re.sub(r"\s+", "_", cc)           # spaces -> underscore
        cc = re.sub(r"[^0-9A-Za-z_]+", "", cc)  # remove non-alphanum/_
        cc = cc.lower()
        rename_map[c] = cc
    df = df.rename(columns=rename_map)

    # For Shapefiles: keep a short alias (<=10 chars) alongside full names
    short_map = {}
    for c in df.columns:
        if len(c) > 10:
            short_map[c] = c[:10]
    return df, short_map

def parse_dates_inplace(df: pd.DataFrame) -> None:
    """Parse any columns that look like dates. Leaves in ISO format yyyy-mm-dd."""
    date_like = [c for c in df.columns if c.endswith('date') or re.search(r'(date|_dt|_ymd|_time)$', c)]
    for c in date_like:
        try:
            parsed = pd.to_datetime(df[c], errors='coerce', infer_datetime_format=True, utc=False)
            # Normalize to date only if time not meaningful
            df[c] = parsed.dt.strftime('%Y-%m-%d')
        except Exception:
            pass

def coerce_numeric(df: pd.DataFrame, cols: Tuple[str, ...]) -> None:
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')

def best_lat_lon_columns(df: pd.DataFrame) -> Tuple[Optional[str], Optional[str]]:
    """Heuristically pick latitude/longitude columns."""
    lat_candidates = [c for c in df.columns if re.search(r'(dec_)?lat', c)]
    lon_candidates = [c for c in df.columns if re.search(r'(dec_)?long|lon', c)]
    for lc in lat_candidates:
        for lo in lon_candidates:
            if lc != lo:
                return lc, lo
    # Common fallbacks
    for pair in [('latitude', 'longitude'), ('lat', 'lon')]:
        if pair[0] in df.columns and pair[1] in df.columns:
            return pair
    return None, None

def datum_to_epsg(datum: Optional[str]) -> int:
    if not datum:
        return 4326  # default WGS84
    d = str(datum).upper()
    if 'NAD83' in d:
        return 4269
    if 'NAD27' in d:
        return 4267
    if 'WGS84' in d or 'WGS 84' in d or d == 'WGS':
        return 4326
    return 4326

def validate_lat_lon(df: pd.DataFrame, lat_col: str, lon_col: str) -> pd.DataFrame:
    """Filter rows to valid geographic ranges and drop obviously bad points."""
    out = df.copy()
    coerce_numeric(out, (lat_col, lon_col))
    before = len(out)
    out = out[(out[lat_col].between(-90, 90)) & (out[lon_col].between(-180, 180))]
    after = len(out)
    print(f"Kept {after}/{before} rows after lat/lon validation.")
    return out

def add_geometry(df: pd.DataFrame, lat_col: str, lon_col: str, epsg_code: int):
    """Return a GeoDataFrame with geometry set from lon/lat in given CRS."""
    import geopandas as gpd
    from shapely.geometry import Point

    gdf = df.copy()
    gdf['geometry'] = [Point(xy) if pd.notna(xy[0]) and pd.notna(xy[1]) else None
                       for xy in zip(gdf[lon_col], gdf[lat_col])]
    gdf = gdf.dropna(subset=['geometry'])
    gdf = gpd.GeoDataFrame(gdf, geometry='geometry', crs=f'EPSG:{epsg_code}')
    return gdf

def export_arcgis(gdf, base_name: str, layer_name: str, short_map: Dict[str, str]):
    """Export to CSV, Shapefile, and GeoPackage for ArcGIS."""
    # CSV (drop geometry to lon/lat columns)
    if EXPORT_CSV:
        csv_path = OUT_DIR / f"{base_name}.csv"
        df_csv = gdf.drop(columns=['geometry']).copy()
        df_csv.to_csv(csv_path, index=False)
        print('Saved CSV:', csv_path)

    # Shapefile
    if EXPORT_SHAPEFILE:
        shp_dir = OUT_DIR / f"{base_name}_shp"
        shp_dir.mkdir(exist_ok=True, parents=True)
        shp_path = shp_dir / f"{base_name}.shp"
        # For shapefile, enforce 10-char column names
        gdf_shp = gdf.copy()
        rename_for_shp = {}
        for full, short in short_map.items():
            if full in gdf_shp.columns:
                # ensure uniqueness
                candidate = short
                i = 1
                while candidate in rename_for_shp.values() or candidate in gdf_shp.columns:
                    candidate = (short[:8] + f"{i:02d}")[:10]
                    i += 1
                rename_for_shp[full] = candidate
        gdf_shp = gdf_shp.rename(columns=rename_for_shp)
        gdf_shp.to_file(shp_path)
        print('Saved Shapefile:', shp_path)

    # GeoPackage
    if EXPORT_GEOPACKAGE:
        gpkg_path = OUT_DIR / f"{base_name}.gpkg"
        gdf.to_file(gpkg_path, layer=layer_name, driver='GPKG')
        print('Saved GeoPackage layer:', gpkg_path, '→', layer_name)

def assign_decade(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
    """Assigns a decade string like '1950s' to each row based on year of date_col."""
    if date_col not in df.columns:
        raise ValueError(f"{date_col} not found in dataframe columns")
    years = pd.to_datetime(df[date_col], errors='coerce').dt.year
    decades = (years // 10) * 10
    df['decade'] = decades.astype('Int64').astype(str) + 's'
    return df

def export_per_decade(gdf, date_col: str, base_prefix: str, layer_prefix: str, shortmap: dict):
    """Export separate GIS layers for each decade in the GeoDataFrame."""
    # Add decade if missing
    if 'decade' not in gdf.columns:
        years = pd.to_datetime(gdf[date_col], errors='coerce').dt.year
        decades = (years // 10) * 10
        gdf = gdf.copy()
        gdf['decade'] = decades.astype('Int64').astype(str) + 's'

    # Group and export
    for dec, sub in gdf.groupby('decade'):
        if pd.isna(dec) or len(sub) == 0:
            continue
        dec_str = str(dec)
        base_name = f"{base_prefix}_{dec_str}"
        layer_name = f"{layer_prefix}_{dec_str}"
        print(f"\nExporting decade {dec_str}: {len(sub)} records → {base_name}")
        export_arcgis(sub, base_name=base_name, layer_name=layer_name, short_map=shortmap)



## 3) Groundwater cleaning
Loads the groundwater sites table, parses dates, validates coordinates, sets CRS from the datum column, and exports.


In [ ]:

# Load
gw = load_tab_file(GROUNDWATER_PATH)
print('Rows in groundwater raw:', len(gw))
display(gw.head(5))

# Standardize names
gw, gw_shortmap = standardize_column_names(gw)

# Parse dates (any *_date, *_dt, etc.)
parse_dates_inplace(gw)

# Pick lat/lon columns
lat_col, lon_col = best_lat_lon_columns(gw)
if not lat_col or not lon_col:
    raise ValueError('Could not identify latitude/longitude columns in groundwater file.')

# Datum → EPSG
datum_col = None
for c in ['dec_coord_datum_cd', 'datum', 'coord_datum', 'geod_datum']:
    if c in gw.columns:
        datum_col = c
        break
epsg_code = datum_to_epsg(gw[datum_col].dropna().iloc[0]) if datum_col else 4326
print('CRS EPSG chosen for GW:', epsg_code, '(from', datum_col or 'default', ')')

# Validate coordinates and numeric types
gw = validate_lat_lon(gw, lat_col, lon_col)

# Optional: keep only site type GW if present
if 'site_tp_cd' in gw.columns:
    before = len(gw)
    gw = gw[gw['site_tp_cd'].str.upper() == 'GW']
    print(f"Filtered GW site_tp_cd → {len(gw)}/{before} rows remain.")

# Make GeoDataFrame
gdf_gw = add_geometry(gw, lat_col, lon_col, epsg_code)

# Export (all records together)
export_arcgis(gdf_gw, base_name='groundwater_sites', layer_name=LAYER_NAME_GW, short_map=gw_shortmap)

# Quick summary
print('\nGroundwater summary:')
print(gdf_gw[[c for c in gdf_gw.columns if c != 'geometry']].describe(include='all', datetime_is_numeric=True))



## 4) Stream discharge cleaning
Same process, but run on the stream discharge table. If time-series data are present, this will still export site points for mapping.


In [ ]:

# Load
sd = load_tab_file(STREAM_PATH)
print('Rows in stream-discharge raw:', len(sd))
display(sd.head(5))

# Standardize names
sd, sd_shortmap = standardize_column_names(sd)

# Parse any date-like columns
parse_dates_inplace(sd)

# Pick lat/lon
lat_col_sd, lon_col_sd = best_lat_lon_columns(sd)
if not lat_col_sd or not lon_col_sd:
    raise ValueError('Could not identify latitude/longitude columns in stream discharge file.')

# Datum → EPSG
datum_col_sd = None
for c in ['dec_coord_datum_cd', 'datum', 'coord_datum', 'geod_datum']:
    if c in sd.columns:
        datum_col_sd = c
        break
epsg_sd = datum_to_epsg(sd[datum_col_sd].dropna().iloc[0]) if datum_col_sd else 4326
print('CRS EPSG chosen for SD:', epsg_sd, '(from', datum_col_sd or 'default', ')')

# Validate coordinates and numeric types
sd = validate_lat_lon(sd, lat_col_sd, lon_col_sd)

# If there's a site type column, keep likely stream gage types (ST, ST-TS, FA, etc.) — adjust as needed
if 'site_tp_cd' in sd.columns:
    before = len(sd)
    sd = sd[sd['site_tp_cd'].str.upper().str.startswith(('ST', 'FA', 'LK', 'OC', 'SB', 'SP', 'SS', 'ST-TS')) | (sd['site_tp_cd'].str.upper() == 'ST')]
    print(f"Filtered stream types → {len(sd)}/{before} rows remain.")

# Build GeoDataFrame
gdf_sd = add_geometry(sd, lat_col_sd, lon_col_sd, epsg_sd)

# Export (all records together)
export_arcgis(gdf_sd, base_name='stream_sites', layer_name=LAYER_NAME_SD, short_map=sd_shortmap)

# Quick summary
print('\nStream discharge summary:')
print(gdf_sd[[c for c in gdf_sd.columns if c != 'geometry']].describe(include='all', datetime_is_numeric=True))



## 5) Add a `decade` column and write CSVs
Creates one large CSV with all records (including `decade`) and separate CSVs for each decade.


In [ ]:

# Try to find a date column in streams
date_candidates = [c for c in gdf_sd.columns if 'date' in c]
if not date_candidates:
    print("No date column detected in stream discharge data; skipping decade CSVs.")
else:
    date_col = date_candidates[0]
    print("Using date column for decade assignment:", date_col)
    gdf_sd = assign_decade(gdf_sd, date_col)

    # Save big file with decade column
    bigfile = OUT_DIR / "stream_sites_with_decades.csv"
    gdf_sd.drop(columns=['geometry']).to_csv(bigfile, index=False)
    print("Saved all-decades CSV:", bigfile)

    # Save per-decade files
    for dec, subset in gdf_sd.groupby('decade'):
        if pd.isna(dec):
            continue
        subfile = OUT_DIR / f"stream_sites_{dec}.csv"
        subset.drop(columns=['geometry']).to_csv(subfile, index=False)
        print("Saved:", subfile)



## 6) Export per-decade GIS layers (Shapefile & GeoPackage)
This will create a Shapefile folder and a GeoPackage layer **for each decade** detected in the data. It runs for **stream discharge** first, then attempts **groundwater** if it has a date column.


In [ ]:

# --- Streams per-decade export ---
date_candidates_sd = [c for c in gdf_sd.columns if 'date' in c]
if date_candidates_sd:
    date_col_sd = date_candidates_sd[0]
    export_per_decade(gdf_sd, date_col=date_col_sd,
                      base_prefix='stream_sites', layer_prefix=LAYER_NAME_SD, shortmap=sd_shortmap)
else:
    print("No date column found for stream data; skipping per-decade GIS export for streams.")

# --- Groundwater per-decade export (if date column present) ---
date_candidates_gw = []
if 'gdf_gw' in globals():
    date_candidates_gw = [c for c in gdf_gw.columns if 'date' in c]
if date_candidates_gw:
    date_col_gw = date_candidates_gw[0]
    export_per_decade(gdf_gw, date_col=date_col_gw,
                      base_prefix='groundwater_sites', layer_prefix=LAYER_NAME_GW, shortmap=gw_shortmap)
else:
    print("No date column found for groundwater data; skipping per-decade GIS export for groundwater.")
